# Advanced Techniques for Language Models

**Mini-Assignment 2**

---

António Cruz (140129), Bruno Santos (140586), Pedro Miranda (129268)


# 1. Introduction

This notebook is the source of the Mini-Assignment 2 report. It documents the alignment stage of the project: starting from the domain-adapted Mini-Assignment 1 model, this notebook adds supervised fine-tuning, then preference alignment via Direct Preference Optimization with reinforcement learning from AI feedback, and compares the three resulting model states on a fixed prompt set.

Mini-Assignment 1 (continued pretraining) is the previous report; the Final Project (system integration) is the next. This notebook assumes the reader has read or has access to the Mini-Assignment 1 report.


## 1.1 Pipeline overview

The full post-training pipeline this project walks through has four stages:

1. Pretraining. A base model is trained on general text by its authors. We start from SmolLM2-360M (open-source).
2. Continued pretraining (Mini-Assignment 1). The base is adapted to the job-postings domain by continued next-token training on raw IT job descriptions. The result is a text completer fluent in the domain.
3. Supervised fine-tuning (Mini-Assignment 2, this stage). The domain-adapted model is taught to follow recruiter instructions, by training on pairs of (query, structured job description).
4. Preference alignment (Mini-Assignment 2, this stage). The supervised model is further trained to prefer outputs that respect constraints, structure and inclusive language, using preferences ranked by a stronger model acting as judge (Gemma-4 in our setup).

Each stage contributes a distinct capability: pretraining gives general language, continued pretraining gives domain fluency, supervised fine-tuning gives instruction-following, and preference alignment shapes the quality of those instruction-following outputs.


# 2. Starting point: the Mini-Assignment 1 checkpoint

Mini-Assignment 1 produced two trained variants of SmolLM2-360M: a full fine-tuning checkpoint and a LoRA adapter. The Mini-Assignment 1 report selected the LoRA variant as the best in-domain fit and the one carried forward.

Before any further training, the LoRA adapter is merged into the base model. The result is a single self-contained model that is equivalent at inference time to (base + Mini-Assignment 1 LoRA), but with no PEFT plumbing. This keeps the rest of the pipeline (supervised fine-tuning, then DPO) simpler: each stage trains a fresh LoRA on top of one consolidated base, with no stacked-adapter bookkeeping.


## 2.1 Merge the Mini-Assignment 1 LoRA into the base

The merge is a one-off operation: load the base model, attach the Mini-Assignment 1 LoRA, call `merge_and_unload`, and save the result. It runs on CPU and takes a few seconds. The base model id is read from the adapter config rather than hardcoded, so the cell stays robust to future model swaps.


In [ ]:
# Merge the Mini-Assignment 1 LoRA into the SmolLM2-360M base, on CPU.
# The merged model is saved to outputs/mp1-360m/merged/ and is the starting
# point for the supervised fine-tuning and DPO stages below.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""   # CPU only; do not touch the GPU.

import json
import time
import gc
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Locate the project root: walk up to a folder containing data/.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

LORA_DIR   = PROJECT_ROOT / "outputs" / "mp1-360m" / "lora"
MERGED_DIR = PROJECT_ROOT / "outputs" / "mp1-360m" / "merged"

# Read the base model id from the adapter config; never hardcoded.
adapter_cfg = json.loads((LORA_DIR / "adapter_config.json").read_text())
BASE_MODEL = adapter_cfg["base_model_name_or_path"]
print(f"Base model:    {BASE_MODEL}")
print(f"LoRA adapter:  {LORA_DIR}")
print(f"Merged output: {MERGED_DIR}")

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16)
model = PeftModel.from_pretrained(model, str(LORA_DIR), torch_dtype=torch.bfloat16)
model = model.merge_and_unload()

MERGED_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

# Free CPU memory before the rest of the notebook loads the model again.
del model, tokenizer
gc.collect()

print(f"Merged in {time.time() - t0:.1f}s.")


## 2.2 Load the merged model

With the merged checkpoint on disk, the rest of the notebook loads the model from `outputs/mp1-360m/merged/`. From here onwards, the model is a regular pretrained-style checkpoint and downstream stages do not need to know about the Mini-Assignment 1 LoRA at all.


In [ ]:
# Load the merged Mini-Assignment 1 model for the rest of the notebook.
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
MERGED_DIR = PROJECT_ROOT / "outputs" / "mp1-360m" / "merged"

ma1_tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)
ma1_model     = AutoModelForCausalLM.from_pretrained(MERGED_DIR, torch_dtype=torch.bfloat16)
total_params  = sum(p.numel() for p in ma1_model.parameters())

print(f"Loaded merged MA1 model from {MERGED_DIR}")
print(f"Total parameters: {total_params:,}")


# 3. Supervised fine-tuning

A continued-pretraining checkpoint is a text completer, not an instruction-follower. Direct Preference Optimization and other alignment techniques assume the starting point already follows instructions, so a short supervised fine-tuning pass on (query, response) pairs is needed before alignment.

This section trains the merged Mini-Assignment 1 model on roughly 7,500 (recruiter query, structured job description) pairs produced by the atlm_teacher ETL agent. The result is a domain-adapted, instruction-following model ready for alignment.


## 3.1 Prompt template

Mini-Assignment 1 left the model as a domain-fluent text completer that has never seen instructions. Before any alignment can happen, the model must be taught to recognise where a prompt ends and where a response should begin, and what kind of response is expected. A consistent prompt template is what carries that signal during supervised fine-tuning.

The template adopted here is Alpaca-inspired, with a one-sentence system framing and two clearly delimited fields:

```
You are a recruitment assistant. Given a brief recruiter request, write a complete structured job posting in Markdown.

### Request
{query}

### Posting
{jd}
```

The same template is used at inference time: the prompt fed to the model is everything up to and including `### Posting\n`, and the model produces the job posting from there.

Three reasons for this choice over the alternatives considered (a leaner key-value template, and a ChatML chat-template setup):

1. Explicit task framing. SmolLM2-360M is small and has never been instruction-tuned. A one-sentence system preamble gives the model an unambiguous anchor for the task, which is worth its roughly 25-token cost.

2. No clash with response content. The structured job descriptions in the training data already use `##` Markdown headings (`## Summary`, `## Required Skills`, `## Responsibilities`, `## Requirements`). The template uses `###` for its own separators, so the model can tell a template marker from a response heading without ambiguity. ChatML-style special tokens would not have this clash, but at the cost of growing the tokenizer's vocabulary.

3. No special-token machinery. Plain-text separators tokenise into existing vocabulary, so no new embeddings need to be learned and no chat template needs to be defined on the tokenizer. The HuggingFace TRL `SFTTrainer` consumes this through a simple formatting function, with no extra configuration.

The supervised fine-tuning stage below uses this template for every one of the roughly 7,500 training pairs.


## 3.2 Training data

The supervised fine-tuning data are the 2,507 records in `data/processed/converted.jsonl`, the output of the atlm_teacher ETL agent that took raw Djinni postings and produced clean, structured job descriptions in Markdown.

Each record has three independent recruiter queries pointing to the same job description. The three queries are fanned out into independent training examples, so the model sees the same target response paired with three different phrasings of the same underlying request. This is a cheap form of augmentation that teaches the model to be robust to phrasing variation, and it roughly triples the effective dataset size.

The records are split 90/10 into train and validation at the record level (not at the query level), so no job description appears in both splits. With a fixed seed of 42, the split is reproducible.


In [ ]:
# Load the SFT data, fan out the three queries per record, format with the
# Section 3.1 template, and split into train/validation at the record level.
import json
import random
from pathlib import Path

from datasets import Dataset

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

SFT_DATA = PROJECT_ROOT / "data" / "processed" / "converted.jsonl"
SEED = 42

# Read raw records (each has `queries` (list of 3) and `job_description`).
records = [json.loads(ln) for ln in open(SFT_DATA, encoding="utf-8")]
print(f"Loaded {len(records):,} records from {SFT_DATA.name}")

# Split at the record level (not the query level), so no JD appears in both
# train and val. 90/10 split, seed 42.
random.Random(SEED).shuffle(records)
n_val = max(1, len(records) // 10)
val_records   = records[:n_val]
train_records = records[n_val:]
print(f"Split: {len(train_records):,} train records, {len(val_records):,} val records")

# Prompt template, exactly as described in Section 3.1.
PROMPT_PREAMBLE = (
    "You are a recruitment assistant. Given a brief recruiter request, "
    "write a complete structured job posting in Markdown."
)

def format_example(query: str, jd: str) -> str:
    return (
        f"{PROMPT_PREAMBLE}\n\n"
        f"### Request\n{query}\n\n"
        f"### Posting\n{jd}"
    )

def fan_out(rs):
    out = []
    for r in rs:
        for q in r["queries"]:
            out.append({"text": format_example(q, r["job_description"])})
    return out

train_examples = fan_out(train_records)
val_examples   = fan_out(val_records)
print(f"After fan-out: {len(train_examples):,} train examples, "
      f"{len(val_examples):,} val examples")

sft_train = Dataset.from_list(train_examples)
sft_val   = Dataset.from_list(val_examples)

# A quick look at one formatted training example, truncated for readability.
print()
print("Sample formatted example:")
print("-" * 60)
print(train_examples[0]["text"][:800])
print("...")


## 3.3 Training configuration

Supervised fine-tuning continues the parameter-efficient approach from Mini-Assignment 1: a fresh LoRA adapter is trained on top of the merged base, leaving the base weights untouched. This keeps both the disk footprint and the GPU memory cost small, and matches the LoRA shape (r=16, alpha=32, dropout=0.05, attention plus feed-forward projections) found to work well in the previous stage.

Hyperparameters mirror Mini-Assignment 1 where applicable: bf16 precision, micro-batch 4 with gradient accumulation 4 (effective batch 16), warmup ratio 0.03, weight decay 0.01, and seed 42. The maximum sequence length is 1024 tokens, which fits every (query, posting) pair without truncation (the longest examples are well under 800 tokens). Training runs for 2 epochs over the roughly 6,800 training examples; with the fanned-out queries each unique job description is seen six times in total (three phrasings, two epochs).

The learning rate is 2e-4, the conventional value for LoRA. The loss is computed over the entire formatted sequence (preamble plus request plus response) rather than only the response; for a small model this is a minor inefficiency, not a correctness issue, and keeps the configuration simple.

The trained adapter, log history and a summary file are written to `outputs/ma2-360m-sft/`. The run tag is `ma2-360m-sft`.


In [ ]:
# Training configuration for the SFT run; consumed by run_sft() below.
SFT_RUN = "ma2-360m-sft"
SFT_OUT = PROJECT_ROOT / "outputs" / SFT_RUN

SFT_CFG = {
    "epochs": 2,
    "per_device_batch": 4,
    "grad_accum": 4,                   # effective batch = 16
    "learning_rate": 2e-4,
    "warmup_ratio": 0.03,
    "weight_decay": 0.01,
    "max_seq_length": 1024,
    # LoRA shape, same as Mini-Assignment 1.
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "lora_target_modules": [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    "seed": SEED,
}
print("SFT config:")
for k, v in SFT_CFG.items():
    print(f"  {k}: {v}")
print(f"output dir: {SFT_OUT}")


## 3.4 Running supervised fine-tuning

The `run_sft` function below wraps the merged Mini-Assignment 1 model in a fresh LoRA, configures the HuggingFace TRL `SFTTrainer`, runs training, and saves the resulting adapter together with a small summary file (trainable parameter count, wall-clock minutes, final validation loss and perplexity). It mirrors the shape of the `run_training` function used in Mini-Assignment 1.

Running the next cell requires a CUDA GPU with bf16 support. On the RTX 4090 used here, the run takes roughly 10 to 15 minutes.


In [ ]:
# Defines run_sft(): one supervised fine-tuning run on top of the merged
# base, producing a new LoRA adapter at outputs/ma2-360m-sft/.
import json
import math
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

def run_sft():
    set_seed(SFT_CFG["seed"])
    SFT_OUT.mkdir(parents=True, exist_ok=True)

    # Load the merged Mini-Assignment 1 model (the SFT starting point).
    tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MERGED_DIR, torch_dtype=torch.bfloat16
    )

    peft_cfg = LoraConfig(
        r=SFT_CFG["lora_r"],
        lora_alpha=SFT_CFG["lora_alpha"],
        lora_dropout=SFT_CFG["lora_dropout"],
        target_modules=SFT_CFG["lora_target_modules"],
        task_type="CAUSAL_LM",
    )

    args = SFTConfig(
        output_dir=str(SFT_OUT),
        num_train_epochs=SFT_CFG["epochs"],
        per_device_train_batch_size=SFT_CFG["per_device_batch"],
        per_device_eval_batch_size=SFT_CFG["per_device_batch"],
        gradient_accumulation_steps=SFT_CFG["grad_accum"],
        learning_rate=SFT_CFG["learning_rate"],
        warmup_ratio=SFT_CFG["warmup_ratio"],
        weight_decay=SFT_CFG["weight_decay"],
        max_length=SFT_CFG["max_seq_length"],
        bf16=True,
        eval_strategy="steps",
        eval_steps=100,
        logging_steps=20,
        save_strategy="no",
        seed=SFT_CFG["seed"],
        report_to=[],
        dataset_text_field="text",
    )

    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=sft_train,
        eval_dataset=sft_val,
        peft_config=peft_cfg,
        processing_class=tokenizer,
    )

    trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in trainer.model.parameters())

    t0 = time.time()
    trainer.train()
    minutes = (time.time() - t0) / 60
    trainer.save_model(str(SFT_OUT))
    tokenizer.save_pretrained(SFT_OUT)

    final = trainer.evaluate()
    summary = {
        "stage": "sft",
        "run": SFT_RUN,
        "learning_rate": SFT_CFG["learning_rate"],
        "trainable_params": trainable,
        "total_params": total,
        "epochs": SFT_CFG["epochs"],
        "train_examples": len(sft_train),
        "val_examples": len(sft_val),
        "minutes": round(minutes, 1),
        "final_eval_loss": round(final["eval_loss"], 4),
        "final_eval_perplexity": round(math.exp(final["eval_loss"]), 2),
    }
    (SFT_OUT / "log_history.json").write_text(
        json.dumps(trainer.state.log_history, indent=1)
    )
    (SFT_OUT / "summary.json").write_text(json.dumps(summary, indent=1))
    print("done:", summary)
    return summary


In [ ]:
# Runs supervised fine-tuning. Needs a CUDA GPU; about 10-15 minutes on the
# RTX 4090 used here.
summary_sft = run_sft()


## 3.5 Sanity check: sample generations

A small set of prompts is run through both the merged Mini-Assignment 1 model (the starting point of this stage) and the supervised fine-tuned model. Side by side, the comparison should show that the SFT model:

1. Recognises the request as an instruction to produce a complete, structured job posting, rather than continuing the input as free text.
2. Produces output with the expected Markdown structure (`## Summary`, `## Required Skills`, `## Responsibilities`, `## Requirements`).
3. Stays on topic with respect to the recruiter query.

This is a qualitative check, not a metric. The full three-way comparison (base, Mini-Assignment 1 plus SFT, aligned) is in Section 6.


In [ ]:
# Quick sanity check: a few prompts through MA1-only (merged) and MA1+SFT,
# greedy decoding for reproducibility.
from peft import PeftModel

SANITY_PROMPTS = [
    "We need a senior data engineer who can design and operate batch and streaming pipelines on AWS.",
    "Looking for a UX designer to lead a small product team and ship a customer-facing dashboard.",
    "We are hiring a DevOps engineer comfortable with Kubernetes, CI/CD, and incident response.",
]

def build_prompt(query: str) -> str:
    return (
        f"{PROMPT_PREAMBLE}\n\n"
        f"### Request\n{query}\n\n"
        f"### Posting\n"
    )

def generate(model, tokenizer, prompt: str, max_new_tokens: int = 400) -> str:
    model.eval()
    enc = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **enc, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(
        out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True
    )

# Load MA1-merged (the starting point) and MA1+SFT.
ma1_tok   = AutoTokenizer.from_pretrained(MERGED_DIR)
ma1       = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR, torch_dtype=torch.bfloat16
).to("cuda")
sft_base  = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR, torch_dtype=torch.bfloat16
)
sft_model = PeftModel.from_pretrained(sft_base, str(SFT_OUT)).to("cuda")

for q in SANITY_PROMPTS:
    p = build_prompt(q)
    print("=" * 80)
    print("REQUEST:", q)
    print("=" * 80)
    print("\n--- merged MA1 (text completer, no SFT) ---")
    print(generate(ma1, ma1_tok, p))
    print("\n--- MA1 + SFT ---")
    print(generate(sft_model, ma1_tok, p))
    print()


# 4. Preference dataset construction

Direct Preference Optimization needs a dataset of (prompt, chosen response, rejected response) triples. Rather than manual annotation or a generic preference dataset, this project builds its own via reinforcement learning from AI feedback: candidates are sampled from the supervised-fine-tuned model and ranked by the atlm_teacher Gemma-4 agent acting as judge.

This section produces the preference dataset that the next section uses for DPO training.


# 5. DPO training

With the supervised-fine-tuned model as both policy and reference, and the preference dataset from Section 4 as supervision, this section runs Direct Preference Optimization to align the model toward the preferred responses. The HuggingFace TRL `DPOTrainer` is used with conventional hyperparameters; the beta (KL coefficient) and learning rate are discussed in the report.


# 6. Three-way evaluation

The final comparison contrasts three model states on the same fixed set of prompts: the untrained base, the Mini-Assignment 1 model after supervised fine-tuning, and the aligned model produced in Section 5. Both automatic metrics (perplexity on a held-out set, LLM-as-judge win-rate against a reference) and qualitative side-by-side examples are reported.


## 6.1 Evaluation prompt set

The three-way evaluation contrasts the untrained base, the supervised fine-tuned model (Section 3) and the aligned model (Section 5) on the same fixed set of 20 recruiter queries. Fixing the set up front, before any further training, removes any temptation to cherry-pick favourable prompts and keeps the comparison honest.

The set is split into two ten-prompt sub-sets that test different things:

1. Held-out in-distribution. Ten queries that match the style and topic mix of the supervised fine-tuning data (`data/processed/converted.jsonl`) but are not part of it. These probe how well each model handles the kind of request it has been trained on; they are the fair comparison for in-domain quality.

2. Fresh out-of-distribution. Ten hand-written queries that probe generalisation: unusual roles, niche tech stacks, atypical phrasings and soft-skill emphasis. These probe how well alignment carries over to prompts that look different from the training distribution, and where it may regress.

The two sub-sets are reported separately throughout Section 6 so the report can comment on each setting. The set is held out both from the supervised fine-tuning data and from the preference-generation prompts that Section 4 draws, so it is the only evaluation input the three models share.


In [ ]:
# Three-way evaluation prompt set: 20 queries, frozen here, used unchanged
# by every comparison in Section 6. Held out from SFT training data and from
# the preference-generation prompts in Section 4.
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

EVAL_PROMPTS = [
    # --- In-distribution: 10 recruiter queries in the style of converted.jsonl ---
    {"id": "ind-01", "subset": "in_distribution",
     "query": "We need a backend engineer to build and maintain Python services on AWS, with hands-on experience in Django and PostgreSQL."},
    {"id": "ind-02", "subset": "in_distribution",
     "query": "Looking for a React developer who can lead frontend architecture on a TypeScript codebase and mentor junior team members."},
    {"id": "ind-03", "subset": "in_distribution",
     "query": "We need an iOS engineer to ship a consumer-facing app in Swift, with experience in SwiftUI and Core Data."},
    {"id": "ind-04", "subset": "in_distribution",
     "query": "Looking for a machine learning engineer to put recommendation models into production, comfortable with PyTorch and Kubernetes."},
    {"id": "ind-05", "subset": "in_distribution",
     "query": "We need a data analyst to build dashboards and explore product data using SQL, dbt and Looker."},
    {"id": "ind-06", "subset": "in_distribution",
     "query": "Looking for a QA automation engineer with Selenium and Playwright experience, to own end-to-end testing of a web platform."},
    {"id": "ind-07", "subset": "in_distribution",
     "query": "We need a cloud engineer to design and operate AWS infrastructure using Terraform, EKS and modern observability tooling."},
    {"id": "ind-08", "subset": "in_distribution",
     "query": "Looking for a full-stack developer comfortable with Node.js, Next.js and PostgreSQL, to ship product features end to end."},
    {"id": "ind-09", "subset": "in_distribution",
     "query": "We need an embedded software engineer to write firmware in C and Rust for low-power IoT devices."},
    {"id": "ind-10", "subset": "in_distribution",
     "query": "Looking for a security engineer to lead application security, threat modelling and code review across multiple product teams."},

    # --- Out-of-distribution: 10 fresh hand-written queries probing generalisation ---
    {"id": "ood-01", "subset": "out_of_distribution",
     "query": "We are hiring a junior developer straight out of a coding bootcamp; the role is heavy on mentorship and pair programming, and the stack is whatever the team is using."},
    {"id": "ood-02", "subset": "out_of_distribution",
     "query": "Looking for a principal staff engineer who has scaled a backend from one to many regions and can act as the org-wide technical authority on distributed systems."},
    {"id": "ood-03", "subset": "out_of_distribution",
     "query": "We need a developer comfortable with Elixir and Phoenix for a real-time messaging product; OTP experience is essential."},
    {"id": "ood-04", "subset": "out_of_distribution",
     "query": "Looking for a quantitative developer to build low-latency trading systems in C++, with a strong background in numerical methods."},
    {"id": "ood-05", "subset": "out_of_distribution",
     "query": "We have a six-month contract for a freelance technical writer who can document a Rust SDK and produce API reference material."},
    {"id": "ood-06", "subset": "out_of_distribution",
     "query": "Looking for someone who is part data engineer and part analytics engineer, comfortable owning the pipeline from ingestion through dbt models to stakeholder dashboards."},
    {"id": "ood-07", "subset": "out_of_distribution",
     "query": "We need a developer who can explain complex topics to non-technical stakeholders, write clear documentation and run customer-facing workshops as part of the role."},
    {"id": "ood-08", "subset": "out_of_distribution",
     "query": "Hiring fully remote across European time zones with an asynchronous-first culture; we need a senior Go engineer to extend our core platform."},
    {"id": "ood-09", "subset": "out_of_distribution",
     "query": "Looking for a game engine programmer comfortable with Unreal Engine 5, gameplay systems and tooling in C++."},
    {"id": "ood-10", "subset": "out_of_distribution",
     "query": "We need a Rust developer to build WebAssembly modules that run client-side in the browser, with experience in performance-critical numerical code."},
]

# Persist as the single source of truth read by the rest of Section 6.
EVAL_DIR  = PROJECT_ROOT / "data" / "processed" / "ma2"
EVAL_DIR.mkdir(parents=True, exist_ok=True)
EVAL_FILE = EVAL_DIR / "eval_prompts.jsonl"
with open(EVAL_FILE, "w", encoding="utf-8") as f:
    for p in EVAL_PROMPTS:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

n_ind = sum(1 for p in EVAL_PROMPTS if p["subset"] == "in_distribution")
n_ood = sum(1 for p in EVAL_PROMPTS if p["subset"] == "out_of_distribution")
print(f"Wrote {len(EVAL_PROMPTS)} evaluation prompts to "
      f"{EVAL_FILE.relative_to(PROJECT_ROOT)}")
print(f"  in_distribution:     {n_ind}")
print(f"  out_of_distribution: {n_ood}")


# 7. Limitations and critical discussion

Alignment frequently regresses capabilities: the model may become more verbose, refuse more often, lose domain knowledge from earlier stages, or develop sycophancy. This section reports honestly where the alignment helped, where it hurt, and what we would do differently. Honest reporting of these regressions is essential, not optional, since they are the trade-off the technique exposes.
